# 04 · Restart, Crash, Grow Up

**Module 08 · Persistent State & Databases**

The last notebook proves the promise of the module: the desk **survives**. Then it looks ahead to what changes when the desk grows.

> No OpenAI calls in this notebook.

In [1]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import db

scratch = Path(tempfile.mkdtemp())
DB_FILE = str(scratch / "desk.db")
db.use_database(DB_FILE)
db.reset_db()
print("Ready")

Ready


## 1. A refund waits for a person, and the person is a different program
In real life the desk and the support lead are **not running at the same time**. The desk puts the refund in the waiting line and may even be shut down. The person opens `approve_pending.py` hours later. They meet in the **database file**.

First, the desk asks for a big refund (this is the agent's tool):

In [2]:
print(db.request_refund("A1004"))

The refund of $250.00 for order A1004 is above the automatic limit of $100. It now waits for a person to approve it.


Now a **separate Python process**, the support lead's tool, which has never heard of this notebook. We point it to the same database file with the `DESK_DB` setting:

In [3]:
env = dict(os.environ)
env["DESK_DB"] = DB_FILE
tool = str(Path("approve_pending.py").resolve())


def run_tool(*arguments):
    result = subprocess.run([sys.executable, tool, *arguments], capture_output=True, text=True, env=env)
    print(result.stdout.strip())


print("--- the support lead lists what is waiting")
run_tool()
print()
print("--- the support lead approves")
run_tool("A1004", "Priya")

--- the support lead lists what is waiting


A1004  $250.00  waiting since 2026-11-22T21:24:26

To decide: python approve_pending.py <order id> <your name> [no]

--- the support lead approves


Refund of $250.00 approved by Priya for order A1004.


In [4]:
# Back in the desk's process: it sees the decision, because the database is the shared truth.
print(db.get_order("A1004")["refund"])
print(db.list_pending())

issued: $250.00 on 2026-11-22T21:24:26
[]


## 2. Pull the plug
`crash_test.py` starts a refund in a **child process** and kills it with `os._exit(1)` (no cleanup at all, like a power cut), once **before** the commit and once **after**. It uses its own temporary database. Here is the whole file's output:

In [5]:
result = subprocess.run([sys.executable, "crash_test.py"], capture_output=True, text=True)
print(result.stdout)

Throwaway database file: crash_test_desk.db (in your temp folder)

Experiment 1: the process died before the commit (exit code 1)
   refunds in the database : 0
   audit lines             : 0
   OK: nothing was half-written. It is as if the refund never started.

Experiment 2: the process died right after the commit (exit code 1)
   refunds in the database : 1
   audit lines             : 1
   OK: the saved refund and its audit line are both there.



- **Before the commit:** nothing was saved, not even the first write. The refund simply never happened.
- **After the commit:** the refund and its audit line are both there.

That is the "A" (atomic) and the "D" (durable) of ACID, shown with a real process killed in the middle.

## 3. Backups
A database is one file, so a backup is a copy. But copying a file **while someone is writing to it** can produce a broken copy. SQLite has a safe way:

In [6]:
import sqlite3

source = sqlite3.connect(DB_FILE)
backup = sqlite3.connect(str(scratch / "backup.db"))
source.backup(backup)                 # a consistent snapshot, even if the desk is running
source.close()

print("Orders in the backup :", backup.execute("SELECT COUNT(*) FROM orders").fetchone()[0])
print("Refunds in the backup:", backup.execute("SELECT COUNT(*) FROM refunds").fetchone()[0])
backup.close()

Orders in the backup : 4
Refunds in the backup: 1


## 4. Changing the tables later (migrations)
Next month we want a `phone` column on customers' tickets. We cannot just wipe the database: it holds real refunds. A **migration** is a small, numbered change that is applied once. SQLite keeps a version number for this in `PRAGMA user_version`:

In [7]:
conn = db.connect()
version = conn.execute("PRAGMA user_version").fetchone()[0]
print("Database version:", version)

if version < 1:
    conn.execute("ALTER TABLE tickets ADD COLUMN phone TEXT")      # migration 1
    conn.execute("PRAGMA user_version = 1")
    conn.commit()
    print("Applied migration 1: tickets.phone")

print("Database version now:", conn.execute("PRAGMA user_version").fetchone()[0])
conn.close()

Database version: 0
Applied migration 1: tickets.phone
Database version now: 1


Every real project does this, usually with a tool (Alembic is the common one in Python). The idea is always the same: **numbered, one-way, applied once, tested on a copy first.**

## 5. When SQLite is not enough
SQLite is a great choice for a laptop, a demo, a small internal tool, or one server. It has limits, and knowing them is part of the answer in an interview:

| Situation | Why SQLite struggles | What teams use |
|---|---|---|
| Several servers share the data | the file lives on **one** machine | **PostgreSQL** (a server everyone connects to) |
| Many writers at once | only **one writer at a time** | PostgreSQL |
| Searching by meaning (RAG at scale) | no vector search built in | PostgreSQL with `pgvector`, or a vector database |
| Very fast shared counters or caches | not its job | **Redis** |

The good news: **everything you learned here carries over**. Transactions, UNIQUE rules, placeholders, audit lines and migrations are the same in PostgreSQL. Only the connection changes.

## What you learned
1. State that must survive goes in a database, not in variables or a JSON file.
2. Transactions make multi-step writes all-or-nothing, and we proved it by killing a process.
3. UNIQUE rules beat check-then-write, and we proved that with eight threads.
4. Memory is useful and also an attack surface.
5. Backups, migrations and the limits of SQLite are part of the job.